# Telecom Upsell Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `upsold`

## 2 · Project Overview

We predict whether a telecom subscriber will accept an **upsell offer** (upgrade to a higher-tier plan) based on their usage patterns, current plan, satisfaction, and tenure.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a subscriber's monthly bill, data usage, call minutes, contract length, support calls, current plan, tenure, and satisfaction score, predict whether they will accept an upsell.

## 5 · Why This Project Matters

- **Revenue growth** through upselling existing customers is more efficient than acquiring new ones.
- Targeting likely upgraders reduces marketing waste.
- Understanding upsell drivers helps design better plan tiers.
- Directly impacts ARPU (average revenue per user).

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | monthly_bill, data_usage_gb, call_minutes, contract_months, tech_support_calls, plan, tenure_months, satisfaction_score |
| **Target** | upsold (0 = no, 1 = yes) |
| **Class balance** | ~60/40 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "upsold"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: upsold
Save dir: E:\Github\Machine-Learning-Projects\Classification\Telecom Upsell Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 telecom subscribers with usage patterns and upsell outcome.

In [4]:
np.random.seed(SEED)
n = 7000
monthly_bill = np.round(np.random.uniform(20, 120, n), 2)
data_usage_gb = np.round(np.random.lognormal(1.5, 0.7, n).clip(0.5, 100), 1)
call_minutes = np.round(np.random.lognormal(5.5, 0.8, n).clip(10, 3000), 0).astype(int)
contract_months = np.random.choice([1, 12, 24], n, p=[0.35, 0.4, 0.25])
tech_support_calls = np.random.poisson(1.2, n).clip(0, 10)
plan = np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.4, 0.35, 0.25])
plan_num = np.where(plan == "Basic", 0, np.where(plan == "Standard", 1, 2))
tenure_months = np.random.randint(1, 72, n)
satisfaction_score = np.random.randint(1, 11, n)

score = (0.01 * data_usage_gb + 0.0005 * call_minutes + 0.005 * monthly_bill
         - 0.3 * plan_num + 0.01 * tenure_months + 0.05 * satisfaction_score
         - 0.1 * tech_support_calls + 0.005 * contract_months
         + np.random.normal(0, 0.7, n))
upsold = (score > np.percentile(score, 60)).astype(int)

df = pd.DataFrame({"monthly_bill": monthly_bill, "data_usage_gb": data_usage_gb,
                    "call_minutes": call_minutes, "contract_months": contract_months,
                    "tech_support_calls": tech_support_calls, "plan": plan,
                    "tenure_months": tenure_months, "satisfaction_score": satisfaction_score,
                    "upsold": upsold})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['upsold'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 9)
Class balance:
upsold
0    0.6
1    0.4
Name: proportion, dtype: float64


,monthly_bill,data_usage_gb,call_minutes,contract_months,tech_support_calls,plan,tenure_months,satisfaction_score,upsold
0,57.45,6.4,361,1,1,Premium,9,5,0
1,115.07,6.9,1320,12,3,Standard,5,9,1
2,93.20,1.9,498,12,1,Standard,11,10,1
3,79.87,1.6,228,24,2,Basic,2,8,0
4,35.60,2.7,245,1,0,Basic,52,9,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 9)

Missing values:
monthly_bill          0
data_usage_gb         0
call_minutes          0
contract_months       0
tech_support_calls    0
plan                  0
tenure_months         0
satisfaction_score    0
upsold                0
dtype: int64

Duplicate rows: 0

Dtypes:
monthly_bill          float64
data_usage_gb         float64
call_minutes            int64
contract_months         int64
tech_support_calls      int32
plan                   object
tenure_months           int32
satisfaction_score      int32
upsold                  int64
dtype: object

Target 'upsold' confirmed. Value counts:
upsold
0    4200
1    2800
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["monthly_bill", "data_usage_gb", "call_minutes", "tenure_months", "satisfaction_score", "tech_support_calls"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Upsell Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("plan")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Upsell Rate by Current Plan")
df.groupby("contract_months")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Upsell Rate by Contract Length")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `upsold`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["salmon", "steelblue"], edgecolor="black")
ax.set_title("Upsell Distribution")
ax.set_xticklabels(["Not Upsold (0)", "Upsold (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Upsell rate: {df[TARGET].mean():.1%}")

Upsell rate: 40.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 8), Test: (1400, 8)
Train class distribution:
upsold
0    0.6
1    0.4
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `plan` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~40% upsold.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["usage_per_dollar"] = X_train["data_usage_gb"] / (X_train["monthly_bill"] + 1)
X_test["usage_per_dollar"] = X_test["data_usage_gb"] / (X_test["monthly_bill"] + 1)

X_train["engagement_index"] = X_train["data_usage_gb"] + X_train["call_minutes"] / 100
X_test["engagement_index"] = X_test["data_usage_gb"] + X_test["call_minutes"] / 100

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['monthly_bill', 'data_usage_gb', 'call_minutes', 'contract_months', 'tech_support_calls', 'plan', 'tenure_months', 'satisfaction_score', 'usage_per_dollar', 'engagement_index']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.6864
F1       : 0.5384

              precision    recall  f1-score   support

           0       0.70      0.84      0.76       840
           1       0.65      0.46      0.54       560

    accuracy                           0.69      1400
   macro avg       0.68      0.65      0.65      1400
weighted avg       0.68      0.69      0.67      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
AdaBoostClassifier          0.694286           0.662500  0.743359  0.685384   0.688797  0.694286    0.241050
NearestCentroid             0.665714           0.661607  0.737744  0.668181   0.673579  0.665714    0.020757
CatBoostClassifier          0.690000           0.660119  0.743287  0.682285   0.684175  0.690000    3.060604
SVC                         0.694286           0.654762  0.735204  0.679808   0.690650  0.694286    1.408860
LGBMClassifier              0.681429           0.653869  0.731178  0.675106   0.675420  0.681429    3.185631
RandomForestClassifier      0.686429           0.652976  0.723764  0.676399   0.680342  0.686429    0.923541
SGDClassifier               0.672143           0.651786  0.722823  0.669557   0.668376  0.6721

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6821
F1: 0.5563


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.5556  (1.6s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[78]	valid_0's binary_logloss: 0.57883
LightGBM F1: 0.5732  (1.2s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               0.6979  0.5732     0.6589  0.5071
FLAML                  0.6821  0.5563     0.6298  0.4982
CatBoost               0.6914  0.5556     0.6553  0.4821
Logistic Regression    0.6864  0.5384     0.6547  0.4571

Top 3 models: ['LightGBM', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  LightGBM:
    Accuracy : 0.6979
    F1       : 0.5732
    Precision: 0.6589
    Recall   : 0.5071

  FLAML:
    Accuracy : 0.6821
    F1       : 0.5563
    Precision: 0.6298
    Recall   : 0.4982

  CatBoost:
    Accuracy : 0.6914
    F1       : 0.5556
    Precision: 0.6553
    Recall   : 0.4821


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: LightGBM

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.82      0.77       840
           1       0.66      0.51      0.57       560

    accuracy                           0.70      1400
   macro avg       0.69      0.67      0.67      1400
weighted avg       0.69      0.70      0.69      1400


Total errors: 423 / 1400 (30.2%)

Sample misclassifications:
      monthly_bill  data_usage_gb  call_minutes  contract_months  tech_support_calls  plan  tenure_months  satisfaction_score  usage_per_dollar  engagement_index  actual  predicted  correct
4064        106.56           12.5           257               24                   0   1.0             70                   2          0.116214             15.07       0          1    False
4229         38.99            7.1           243               12                   3   0.0             21                  10          0.177544              9.53       1          0    Fa

## 25 · Interpretation and Business Insight

**Key findings:**
- **Basic plan** subscribers have the highest upsell rate (room to upgrade).
- **Data usage** is the strongest upsell predictor (power users need more).
- **Satisfaction score** correlates positively with upsell acceptance.
- **Tech support calls** negatively correlate (frustrated users don't upgrade).

**Business takeaway:** Target Basic plan subscribers with high data usage and high satisfaction scores.

## 26 · Limitations

1. Synthetic data with simplified upsell dynamics.
2. No offer details (discount, timing, channel).
3. Missing competitive alternatives.
4. No A/B test data on different offers.
5. Binary outcome ignores partial upgrades.

## 27 · How to Improve This Project

1. Use real CRM data with upsell campaign results.
2. Add offer details (price, timing, channel).
3. Include competitive intelligence.
4. Model uplift (causal effect of offer) rather than just propensity.
5. Add network quality metrics by region.

## 28 · Production Considerations

- Integrate with CRM for targeted campaign execution.
- Output upsell probability for offer prioritization.
- A/B test different offer types per segment.
- Monitor upsell rates and model accuracy monthly.
- Combine with churn prediction (don't upsell at-risk customers).

## 29 · Common Mistakes

1. Upselling dissatisfied customers (increases churn).
2. Not considering current plan in feature engineering.
3. Using accuracy on imbalanced upsell data.
4. Ignoring the timing of the offer.
5. Not measuring incremental lift from the model vs. random targeting.

## 30 · Mini Challenge / Exercises

1. Remove `plan` — how much does F1 change?
2. Segment by plan tier and build separate models.
3. Create `headroom = max_plan_value - monthly_bill` proxy.
4. Plot upsell rate by satisfaction score deciles.
5. Try uplift modeling (treated vs. control) conceptually.

## 31 · Final Summary / Key Takeaways

1. **Current plan tier** is the strongest upsell predictor (Basic → upgrade).
2. **High data usage** signals need for premium plans.
3. **Satisfaction and tenure** indicate willingness to commit.
4. **Tech support issues** reduce upsell likelihood.
5. **Targeted upselling** outperforms mass campaigns.
6. **Uplift modeling** would further optimize by measuring causal impact.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Telecom Upsell Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.6914,
    "f1": 0.5556,
    "precision": 0.6553,
    "recall": 0.4821
  },
  "LightGBM": {
    "accuracy": 0.6979,
    "f1": 0.5732,
    "precision": 0.6589,
    "recall": 0.5071
  },
  "Logistic Regression": {
    "accuracy": 0.6864,
    "f1": 0.5384,
    "precision": 0.6547,
    "recall": 0.4571
  },
  "FLAML": {
    "accuracy": 0.6821,
    "f1": 0.5563,
    "precision": 0.6298,
    "recall": 0.4982
  }
}
